## Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

## Load Preprocessed Dataset

In [ ]:
processed_path = "../dataset/processed"
df = pd.read_parquet(processed_path)

## Map target to binary

In [ ]:
df['default_flag'] = df['loan_status'].apply(lambda x: 1 if x == 'Charged Off' else 0)

## Define features

In [ ]:
numeric_features = ["revol_util", "dti", "total_bal_ex_mort", "num_accts_ever_120_pd", "delinq_2yrs", "annual_inc"]
categorical_features = ["home_ownership", "purpose", "grade", "sub_grade", "term", "verification_status"]

df[numeric_features] = df[numeric_features].fillna(0.0)

## Preprocessing

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

X = preprocessor.fit_transform(df[numeric_features + categorical_features])
y = df['default_flag'].values

## Handle Class Imbalance (Undersample Majority)

In [ ]:
df_full = pd.DataFrame(X)
df_full['target'] = y

# Separate majority and minority
df_majority = df_full[df_full.target == 0]
df_minority = df_full[df_full.target == 1]

# Undersample majority
df_majority_downsampled = resample(df_majority,
                                   replace=False,
                                   n_samples=len(df_minority)*100,  # keep 100x minority
                                   random_state=42)

df_balanced = pd.concat([df_majority_downsampled, df_minority])

X_balanced = df_balanced.drop('target', axis=1).values
y_balanced = df_balanced['target'].values

## Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train target distribution: {np.bincount(y_train)}, Test target distribution: {np.bincount(y_test)}")

## Build Neural Network

In [ ]:
input_dim = X_train.shape[1]

model = Sequential([
    Dense(128, input_dim=input_dim, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])

early_stop = EarlyStopping(monitor='val_auc', patience=5, mode='max', restore_best_weights=True)

## Train the model

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    class_weight={0: 1, 1: 100},  # emphasize minority class
    callbacks=[early_stop],
    verbose=1
)

## Evaluate the model

In [ ]:
results = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {results[0]:.4f}, Test AUC: {results[1]:.4f}")

## Predict probabilities

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred_class = (y_pred_prob >= 0.5).astype(int)

## Save the trained model (Keras 3 compatible)

In [ ]:
# Option 1: Keras native format (.keras) — recommended
model.save("lending_club_nn_model.keras")
print("Model saved as 'lending_club_nn_model.keras'.")

# Option 2: HDF5 format (.h5)
model.save("lending_club_nn_model.h5")
print("Model saved as 'lending_club_nn_model.h5'.")
